In [ ]:
import numpy as np
import pandas as pd
from statsmodels.duration.hazard_regression import PHReg
from statsmodels.tools.sm_exceptions import PerfectSeparationError, ConvergenceWarning
import warnings
import matplotlib.pyplot as plt

In [ ]:
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
def fp_transform(t, powers):
    """
    Compute fractional polynomial transformations for time t.
    Returns a list of transformation terms.
    For FP1 (len(powers)=1): [t^{p1}]
    For FP2 (len(powers)=2): [t^{p1}, t^{p2}] if p1 != p2, else [t^{p1}, t^{p1} * log(t)]
    """
    t = np.maximum(t, 1e-5)  # Avoid t=0 or negative
    p_sorted = sorted(powers)  # Sort to standardize
    if len(powers) == 1:  # FP1
        p1 = p_sorted[0]
        return [np.power(t, p1)]
    else:  # FP2
        p1, p2 = p_sorted
        if np.isclose(p1, p2):
            term1 = np.power(t, p1)
            term2 = term1 * np.log(t)
        else:
            term1 = np.power(t, p1)
            term2 = np.power(t, p2)
        return [term1, term2]

In [ ]:
def fitness(powers, df, covariate_name='X', time_name='time', event_name='event'):
    """
    Fitness function: Fit Cox PH model with FP time interaction and return -AIC.
    Higher value is better.
    """
    t = df[time_name].values
    event = df[event_name].values
    X = df[covariate_name].values
    endog = np.column_stack((t, event))  # PHReg expects [time, event]

    fp_terms = fp_transform(t, powers)
    exog_data = {'X': X}
    for i, term in enumerate(fp_terms, 1):
        exog_data[f'X_fp{i}'] = X * term
    exog = pd.DataFrame(exog_data)
    k = len(fp_terms) + 1  # df: beta_X + beta_fp1 (+ beta_fp2 if FP2)

    try:
        model = PHReg(endog=endog, exog=exog)
        res = model.fit()
        log_lik = res.llf
        aic = -2 * log_lik + 2 * k
        return -aic  # Maximize by returning negative AIC
    except (PerfectSeparationError, ValueError, np.linalg.LinAlgError):
        return -np.inf  # Invalid model

In [ ]:
def tournament_selection(pop, fits, tournament_size=3):
    """
    Tournament selection: Select the best from a random subset.
    """
    selected = []
    for _ in range(len(pop)):
        competitors = np.random.choice(len(pop), tournament_size)
        best = competitors[np.argmax([fits[i] for i in competitors])]
        selected.append(pop[best])
    return selected

In [ ]:
def crossover(parent1, parent2, crossover_rate=0.7):
    """
    Arithmetic crossover for arrays.
    """
    if np.random.rand() < crossover_rate:
        alpha = np.random.uniform(0, 1)
        child1 = alpha * parent1 + (1 - alpha) * parent2
        child2 = (1 - alpha) * parent1 + alpha * parent2
        return child1, child2
    else:
        return parent1, parent2

In [ ]:
def genetic_algorithm(df, fp_degree=2, pop_size=50, generations=100, covariate_name='X', time_name='time', event_name='event',
                      elitism_rate=0.1, crossover_rate=0.7, mutation_rate=0.1, bounds=[-3, 3]):
    """
    Genetic Algorithm to optimize FP powers for time-varying effects in Cox model.
    """
    # Initialize population as list of arrays
    pop = [np.random.uniform(bounds[0], bounds[1], fp_degree) for _ in range(pop_size)]

    best_fits = []  # Track best fitness over generations

    for gen in range(generations):
        # Evaluate fitness
        fits = [fitness(ind, df, covariate_name, time_name, event_name) for ind in pop]

        # Track best
        best_idx = np.argmax(fits)
        best_fits.append(fits[best_idx])
        print(f"Generation {gen}: Best fitness = {fits[best_idx]}, Powers = {pop[best_idx]}")  # For monitoring

        # Elitism: Carry over top elites
        elite_count = int(elitism_rate * pop_size)
        elites = [pop[i] for i in np.argsort(fits)[-elite_count:]]

        # Selection
        selected = tournament_selection(pop, fits)

        # Crossover and mutation to create new population
        new_pop = elites[:]  # Start with elites
        while len(new_pop) < pop_size:
            parent1_idx = np.random.randint(0, len(selected))
            parent2_idx = np.random.randint(0, len(selected))
            child1, child2 = crossover(selected[parent1_idx], selected[parent2_idx], crossover_rate)
            child1 = mutate(child1, mutation_rate, bounds=bounds)
            child2 = mutate(child2, mutation_rate, bounds=bounds)
            new_pop.append(child1)
            if len(new_pop) < pop_size:
                new_pop.append(child2)

        pop = new_pop[:pop_size]  # Ensure size

    # Return best
    best_idx = np.argmax(fits)
    return pop[best_idx], fits[best_idx], best_fits

In [ ]:
if __name__ == "__main__":
    # Generate synthetic survival data (Weibull distributed times, binary covariate)
    np.random.seed(42)
    n = 200
    X = np.random.binomial(1, 0.5, n)  # Binary covariate
    # Time-varying effect: hazard = exp(-1 + 0.5*X + 0.2 * log(t) * X)
    lambda_ = 0.01  # Scale
    shape = 1.5  # Weibull shape
    u = np.random.uniform(0, 1, n)
    t = (-np.log(u) / (lambda_ * np.exp(-1 + 0.5 * X))) ** (1 / shape)  # Base times
    # Censoring
    censor_time = np.random.uniform(5, 20, n)
    observed_time = np.minimum(t, censor_time)
    event = (t <= censor_time).astype(int)

    df = pd.DataFrame({'time': observed_time, 'event': event, 'X': X})

    # Run GA for FP2
    best_powers, best_fit, fitness_history = genetic_algorithm(df, fp_degree=2, generations=50, pop_size=30)

    print(f"Best powers: {best_powers}")
    print(f"Best fitness: {best_fit}")

    # Convergence visualization
    plt.figure(figsize=(8, 5))
    plt.plot(range(len(fitness_history)), fitness_history, marker='o', linestyle='-')
    plt.xlabel('Generation')
    plt.ylabel('Best Fitness (-AIC)')
    plt.title('Genetic Algorithm Convergence')
    plt.grid(True)
    plt.show()